In [1]:
# packages
import pandas as pd
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from sklearn.neighbors import NearestNeighbors
import json

In [2]:
# set model
model = "bert-base"

# set layer
layer = "layer_12"

# set column name?

# set file
sample = pd.read_parquet("/kaggle/input/datasets/lianestrauch/bert-base-samples/sample_bert_base_last.parquet")

# set eps
eps_values = {
    5: np.round(np.arange(0.05, 0.17 + 0.01, 0.01), 2),
    10: np.round(np.arange(0.06, 0.18 + 0.01, 0.01), 2),
    50: np.round(np.arange(0.07, 0.19 + 0.01, 0.01), 2),
    100: np.round(np.arange(0.08, 0.19 + 0.01, 0.01), 2),
}

# set outputfile
output_file = Path(f"clustering_results_{model}_{layer}.json")


In [3]:
SAMPLE_PATH = Path("/kaggle/input/datasets/lianestrauch/bert-base-samples")

In [4]:
!pip install kDBCV

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 927.5 kB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.8/60.8 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 52.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.8/40.8 MB 34.5 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
  Attempting uninstall: scipy
    Found existing installation: scipy 1.16.3
    Uninstalling scipy-1.16.3:
      Successfully uninstalled scipy-1.16.3
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
bigframes 2.39.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
kaggle-environments 1.29.3 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
cesium 0.12.4 

In [5]:
import json
import time
from pathlib import Path
import numpy as np

# Patch NumPy 2.0 compatibility for legacy libraries like kDBCV
if not hasattr(np, 'float_'):
    np.float_ = np.float64
if not hasattr(np, 'int_'):
    np.int_ = np.int64

from kDBCV import DBCV_score
from scipy.spatial.distance import cosine
from sklearn.preprocessing import normalize

from sklearn.cluster import DBSCAN

# ============================================================
# Load existing results, or create a new dictionary
# ============================================================

if output_file.exists():
    with open(output_file, "r") as f:
        clustering_results = json.load(f)

    print(
        f"Loaded {len(clustering_results)} existing clustering results."
    )
else:
    clustering_results = {}

    print("No existing results found. Starting a new file.")


# ============================================================
# Prepare embeddings
# ============================================================

col_name = sample.columns[-1]

embeddings = np.vstack(sample[col_name].values)
n_samples = len(embeddings)

print(f"Number of data points: {n_samples}")


# ============================================================
# Run clustering
# ============================================================

for min_samples, current_eps_values in eps_values.items():

    for eps in current_eps_values:

        key = f"eps_{eps:.4f}_minPts_{min_samples}"

        # ----------------------------------------------------
        # Skip if this combination has already been calculated
        # ----------------------------------------------------
        if key in clustering_results:
            print(
                f"SKIPPING: eps={eps:.4f}, "
                f"minPts={min_samples} "
                f"(already calculated)"
            )
            continue

        print(
            f"Running: eps={eps:.4f}, "
            f"minPts={min_samples}..."
        )

        start_time = time.perf_counter()

        labels = DBSCAN(
            eps=eps,
            min_samples=min_samples,
            metric="cosine"
        ).fit_predict(embeddings)

        elapsed_time = time.perf_counter() - start_time

        # ----------------------------------------------------
        # Cluster statistics
        # ----------------------------------------------------

        unique_labels, counts = np.unique(
            labels,
            return_counts=True
        )

        # Exclude noise (-1)
        cluster_counts = counts[unique_labels != -1]

        n_clusters = len(cluster_counts)
        n_noise = int(np.sum(labels == -1))

        # Largest cluster
        if len(cluster_counts) > 0:
            largest_cluster_size = int(np.max(cluster_counts))
        else:
            largest_cluster_size = 0

        # Percentage of full dataset
        largest_cluster_pct = (
            100 * largest_cluster_size / n_samples
            if n_samples > 0 else 0
        )

        # DBCV
        embeddings_norm = normalize(embeddings, norm='l2', axis=1) # dbcv only takes euclidean distance
        score = DBCV_score(embeddings_norm, labels)
        print("DBCV Score (based on normalised embeddings and euclidean distance):", score)

        # ----------------------------------------------------
        # Store result
        # ----------------------------------------------------

        clustering_results[key] = {
            "model": model,
            "layer": layer,
            "eps": float(eps),
            "min_samples": int(min_samples),
            "n_clusters": int(n_clusters),
            "n_noise": n_noise,
            "largest_cluster_size": largest_cluster_size,
            "largest_cluster_pct": float(largest_cluster_pct),
            "runtime_seconds": float(elapsed_time),
            "n_samples": int(n_samples),
            "dbcv": score,
            "labels": { str(sample_id): int(label) for sample_id, label in zip(sample["id"], labels) }
        }

        print(
            f"  finished: "
            f"{n_clusters} clusters, "
            f"largest={largest_cluster_size} "
            f"({largest_cluster_pct:.2f}%), "
            f"time={elapsed_time:.2f}s"
        )

        # ----------------------------------------------------
        # Save immediately after each clustering
        # ----------------------------------------------------
        #
        # This is useful for long-running experiments:
        # if the script crashes halfway through, everything
        # completed so far is already saved.
        #

        with open(output_file, "w") as f:
            json.dump(clustering_results, f)


# ============================================================
# Done
# ============================================================

print(
    f"\nDone. Total stored results: "
    f"{len(clustering_results)}"
)

No existing results found. Starting a new file.
Number of data points: 49919
Running: eps=0.0500, minPts=5...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(0.0006784707711786198), None)
  finished: 12 clusters, largest=173 (0.35%), time=55.95s
Running: eps=0.0600, minPts=5...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(0.0016002369527538913), None)
  finished: 19 clusters, largest=319 (0.64%), time=55.82s
Running: eps=0.0700, minPts=5...


...

DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(0.009413912812899211), None)
  finished: 4 clusters, largest=2615 (5.24%), time=60.06s
Running: eps=0.1300, minPts=100...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(0.02005804348976185), None)
  finished: 2 clusters, largest=7156 (14.34%), time=55.69s
Running: eps=0.1400, minPts=100...


/usr/local/lib/python3.12/dist-packages/kDBCV/DBCV.py:359: RuntimeWarning: overflow encountered in power
  distance_matrix_condensed = (1 / distance_matrix_condensed)**d
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:52: RuntimeWarning: overflow encountered in reduce


DBCV Score (based on normalised embeddings and euclidean distance): (np.float64(-0.006442182816941614), None)
  finished: 3 clusters, largest=13617 (27.28%), time=55.85s
Running: eps=0.1500, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 2 clusters, largest=21579 (43.23%), time=55.78s
Running: eps=0.1600, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 3 clusters, largest=28909 (57.91%), time=55.80s
Running: eps=0.1700, minPts=100...
memory cutoff reached
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 2 clusters, largest=36123 (72.36%), time=56.22s
Running: eps=0.1800, minPts=100...
Not enough clusters: must have at least two.
DBCV Score (based on normalised embeddings and euclidean distance): (-1, None)
  finished: 1 clusters, largest=41312 (82.76%), time=56.16s
Running: eps=0.1900, mi